# Imports

In [1]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

con = duckdb.connect()

FEATURE_PATH = "data/processed/CA_1_features_v1.parquet"

# Inspect date range

In [2]:
con.execute(f"""
SELECT
    MIN(date) AS min_date,
    MAX(date) AS max_date,
    COUNT(DISTINCT date) AS n_dates
FROM read_parquet('{FEATURE_PATH}')
""").fetchdf()

,min_date,max_date,n_dates
0,2011-01-29,2016-05-22,1941


# Use the last 28 days as test and previous 28 as validation

In [3]:
split_info = con.execute(f"""
SELECT
    MAX(date) AS max_date
FROM read_parquet('{FEATURE_PATH}')
""").fetchdf()

max_date = pd.to_datetime(split_info.loc[0, "max_date"])

test_start = max_date - pd.Timedelta(days=27)
val_start = test_start - pd.Timedelta(days=28)

print("Validation starts:", val_start)
print("Test starts:", test_start)
print("Final date:", max_date)

Validation starts: 2016-03-28 00:00:00
Test starts: 2016-04-25 00:00:00
Final date: 2016-05-22 00:00:00


# Verify split sizes

In [4]:
con.execute(f"""
SELECT
    CASE
        WHEN date < DATE '{val_start.date()}' THEN 'train'
        WHEN date < DATE '{test_start.date()}' THEN 'validation'
        ELSE 'test'
    END AS split,

    COUNT(*) AS rows,
    COUNT(DISTINCT date) AS days

FROM read_parquet('{FEATURE_PATH}')

GROUP BY split
ORDER BY split
""").fetchdf()

,split,rows,days
0,test,85372,28
1,train,5747365,1885
2,validation,85372,28


# Filter unavailable products

In [5]:
con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN is_available = 1 THEN 1 ELSE 0 END) AS available_rows
FROM read_parquet('{FEATURE_PATH}')
""").fetchdf()

,total_rows,available_rows
0,5918109,4788267.0


# Baseline 1 — yesterday's demand

In [6]:
baseline_1 = con.execute(f"""
SELECT
    units_sold,
    lag_1 AS prediction
FROM read_parquet('{FEATURE_PATH}')
WHERE
    date >= DATE '{val_start.date()}'
    AND date < DATE '{test_start.date()}'
    AND is_available = 1
    AND lag_1 IS NOT NULL
""").fetchdf()

In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(
    baseline_1["units_sold"],
    baseline_1["prediction"]
)

rmse = np.sqrt(
    mean_squared_error(
        baseline_1["units_sold"],
        baseline_1["prediction"]
    )
)

print("Lag-1 MAE:", mae)
print("Lag-1 RMSE:", rmse)

Lag-1 MAE: 1.2817317153164973
Lag-1 RMSE: 2.768170875297637


# Baseline 2 — same day last week

In [8]:
baseline_7 = con.execute(f"""
SELECT
    units_sold,
    lag_7 AS prediction
FROM read_parquet('{FEATURE_PATH}')
WHERE
    date >= DATE '{val_start.date()}'
    AND date < DATE '{test_start.date()}'
    AND is_available = 1
    AND lag_7 IS NOT NULL
""").fetchdf()

In [9]:
mae_7 = mean_absolute_error(
    baseline_7["units_sold"],
    baseline_7["prediction"]
)

rmse_7 = np.sqrt(
    mean_squared_error(
        baseline_7["units_sold"],
        baseline_7["prediction"]
    )
)

print("Lag-7 MAE:", mae_7)
print("Lag-7 RMSE:", rmse_7)

Lag-7 MAE: 1.3096799887550954
Lag-7 RMSE: 2.8065592077338195


# Baseline 3 — 28-day rolling mean

In [10]:
baseline_roll = con.execute(f"""
SELECT
    units_sold,
    rolling_mean_28 AS prediction
FROM read_parquet('{FEATURE_PATH}')
WHERE
    date >= DATE '{val_start.date()}'
    AND date < DATE '{test_start.date()}'
    AND is_available = 1
    AND rolling_mean_28 IS NOT NULL
""").fetchdf()

In [11]:
mae_roll = mean_absolute_error(
    baseline_roll["units_sold"],
    baseline_roll["prediction"]
)

rmse_roll = np.sqrt(
    mean_squared_error(
        baseline_roll["units_sold"],
        baseline_roll["prediction"]
    )
)

print("Rolling-28 MAE:", mae_roll)
print("Rolling-28 RMSE:", rmse_roll)

Rolling-28 MAE: 1.087551706481215
Rolling-28 RMSE: 2.193423582379451


# Compare them

In [12]:
results = pd.DataFrame({
    "model": [
        "Lag 1",
        "Lag 7",
        "Rolling Mean 28"
    ],
    "MAE": [
        mae,
        mae_7,
        mae_roll
    ],
    "RMSE": [
        rmse,
        rmse_7,
        rmse_roll
    ]
})

results.sort_values("MAE")

,model,MAE,RMSE
2,Rolling Mean 28,1.087552,2.193424
0,Lag 1,1.281732,2.768171
1,Lag 7,1.309680,2.806559
